In [ ]:
#!/usr/bin/env python3
"""
Interactive Color Matching using the exact Bayesian Optimization implementation
- Allows user to choose a target color
- Uses the exact same BO implementation from the original code
- Prints CMYK values after each iteration
"""

from pyDOE import lhs
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import distance
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from scipy.optimize import minimize
from scipy.stats import norm
import random

# Color conversion functions
def RGB_to_CMYK(RGB_sample):
    Red = RGB_sample[0]
    Green = RGB_sample[1]
    Blue = RGB_sample[2]

    Black = min(1-Red, 1-Green, 1-Blue)
    
    # Handle division by zero
    if Black == 1:
        return np.array([0, 0, 0, 1])
        
    Cyan = (1-Red-Black)/(1-Black)
    Magenta = (1-Green-Black)/(1-Black)
    Yellow = (1-Blue-Black)/(1-Black) 

    return np.array([Cyan, Magenta, Yellow, Black])

def CMYK_to_RGB(cmyk):
    """Convert CMYK in [0,1] to RGB in [0,1]"""
    c, m, y, k = cmyk
    r = (1 - c) * (1 - k)
    g = (1 - m) * (1 - k)
    b = (1 - y) * (1 - k)
    return np.array([r, g, b])

def get_target_color():
    """Let user choose a target color"""
    print("\n=== Target Color Selection ===")
    print("Enter RGB values (0-255) for your target color")
    
    # Option to use a predefined color
    preset_option = input("Use a preset color? (y/n): ").strip().lower()
    if preset_option == 'y':
        presets = {
            '1': [255, 0, 0],    # Red
            '2': [0, 255, 0],    # Green
            '3': [0, 0, 255],    # Blue
            '4': [255, 255, 0],  # Yellow
            '5': [0, 255, 255],  # Cyan
            '6': [255, 0, 255],  # Magenta
            '7': [128, 0, 128],  # Purple
            '8': [165, 42, 42],  # Brown
            '9': [255, 165, 0],  # Orange
            '10': [75, 46, 131], # Deep Purple (from your example)
        }
        
        print("Available preset colors:")
        for key, color in presets.items():
            print(f"{key}: RGB{tuple(color)}")
            
        choice = input("Enter your choice (1-10): ").strip()
        if choice in presets:
            target_rgb = presets[choice]
        else:
            print("Invalid choice, using default color (75, 46, 131)")
            target_rgb = [75, 46, 131]
    else:
        # Manual RGB entry
        try:
            r = int(input("Red (0-255): ").strip())
            g = int(input("Green (0-255): ").strip())
            b = int(input("Blue (0-255): ").strip())
            target_rgb = [max(0, min(255, r)), max(0, min(255, g)), max(0, min(255, b))]
        except ValueError:
            print("Invalid input, using default color (75, 46, 131)")
            target_rgb = [75, 46, 131]
    
    # Normalize to [0,1] range
    max_RGB = 255
    target_rgb_norm = np.array(target_rgb) / max_RGB
    target_cmyk = RGB_to_CMYK(target_rgb_norm)
    
    print(f"\nTarget color: RGB{target_rgb} → CMYK[{', '.join([f'{v:.3f}' for v in target_cmyk])}]")
    
    # Display the target color
    plt.figure(figsize=(3, 3))
    plt.imshow([[target_rgb_norm]], extent=[0, 1, 0, 1])
    plt.axis("off")
    plt.title("Target Color")
    plt.show()
    
    return target_rgb, target_rgb_norm, target_cmyk

def get_initial_sample():
    """Get initial CMYK sample"""
    print("\n=== Initial Sample Selection ===")
    print("Enter initial CMYK values (0-270) or use random")
    
    use_random = input("Use random initial sample? (y/n): ").strip().lower()
    
    max_well_volume = 270
    
    if use_random == 'n':
        try:
            c = float(input("Cyan (0-270): ").strip())
            m = float(input("Magenta (0-270): ").strip())
            y = float(input("Yellow (0-270): ").strip())
            k = float(input("Black (0-270): ").strip())
            
            # Validate input
            c = max(0, min(270, c))
            m = max(0, min(270, m))
            y = max(0, min(270, y))
            k = max(0, min(270, k))
            
            # Normalize to [0,1] range
            sample_CMYK_norm = np.array([[c, m, y, k]]) / max_well_volume
        except ValueError:
            print("Invalid input, using random sample")
            sample_CMYK_norm = np.random.rand(1, 4)
    else:
        print("Generating random CMYK sample...")
        sample_CMYK_norm = np.random.rand(1, 4)
        
        # Scale values to make them more reasonable
        total = np.sum(sample_CMYK_norm)
        if total > 2.0:  # Arbitrary threshold
            sample_CMYK_norm = sample_CMYK_norm * (2.0 / total)
    
    print(f"\nInitial CMYK sample: [{', '.join([f'{v:.3f}' for v in sample_CMYK_norm[0]])}]")
    initial_rgb = CMYK_to_RGB(sample_CMYK_norm[0])
    
    # Display the initial color
    plt.figure(figsize=(3, 3))
    plt.imshow([[initial_rgb]], extent=[0, 1, 0, 1])
    plt.axis("off")
    plt.title("Initial Sample Color")
    plt.show()
    
    return sample_CMYK_norm

def display_cmyk_volumes(cmyk, max_volume=270, title="CMYK Volumes"):
    """Display CMYK volumes in microliters"""
    volumes = [round(val * max_volume, 2) for val in cmyk]
    
    print(f"\n=== {title} ===")
    print(f"Cyan: {volumes[0]}µL")
    print(f"Magenta: {volumes[1]}µL")
    print(f"Yellow: {volumes[2]}µL")
    print(f"Black: {volumes[3]}µL")
    print(f"Total: {sum(volumes):.2f}µL")
    
    return volumes

def compare_colors(cmyk1, cmyk2, labels=["Color 1", "Color 2"]):
    """Compare two CMYK colors side by side"""
    rgb1 = CMYK_to_RGB(cmyk1)
    rgb2 = CMYK_to_RGB(cmyk2)
    
    plt.figure(figsize=(6, 3))
    plt.subplot(1, 2, 1)
    plt.imshow([[rgb1]], extent=[0, 1, 0, 1])
    plt.axis("off")
    plt.title(labels[0])
    
    plt.subplot(1, 2, 2)
    plt.imshow([[rgb2]], extent=[0, 1, 0, 1])
    plt.axis("off")
    plt.title(labels[1])
    
    plt.tight_layout()
    plt.show()

def column(matrix, i):
    """Extract column i from matrix"""
    return np.array([row[i] for row in matrix])

def visualize_color_selection(X_samples, target_color_norm):
    """
    Visualize the color combinations chosen by the algorithm over iterations.
    Display colors in a grid with 10 colors per row.
    """
    n_samples = len(X_samples)
    colors_per_row = 5  # Limit to 5 colors per row
    n_rows = (n_samples + colors_per_row - 1) // colors_per_row  # Calculate total rows

    plt.figure(figsize=(12, n_rows * 2))
    for idx, cmyk in enumerate(X_samples):
        row = idx // colors_per_row
        col = idx % colors_per_row
        
        # Convert CMYK to RGB for display
        rgb = CMYK_to_RGB(cmyk)

        # Each color is displayed in its own grid
        plt.subplot(n_rows, colors_per_row, idx + 1)
        plt.imshow([[rgb]], extent=[0, 1, 0, 1])
        plt.axis("off")
        plt.title(f"Iter {idx + 1}", fontsize=10)

    plt.tight_layout()
    plt.show()
    
    # Show target color for reference
    plt.figure(figsize=(3, 3))
    target_rgb = CMYK_to_RGB(target_color_norm)
    plt.imshow([[target_rgb]], extent=[0, 1, 0, 1])
    plt.axis("off")
    plt.title("Target Color", fontsize=12)
    plt.show()

def run_interactive_bo_simulation():
    """Run the interactive BO simulation using the exact same algorithm"""
    # Constants
    max_well_volume = 270
    hyperparameter = 0.01
    threshold = 0.025
    max_iterations = 20
    n_features = 4  # CMYK components
    bounds = [(0, 1)] * n_features  # CMYK values are normalized between 0 and 1
    
    # Step 1: Get target color from user
    target_rgb, target_rgb_norm, target_cmyk = get_target_color()
    
    # Step 2: Get initial sample
    sample_CMYK_norm = get_initial_sample()
    
    # Step 3: Calculate initial distance and objective
    initial_distance = np.linalg.norm(sample_CMYK_norm - target_cmyk)
    print(f"Initial distance to target: {initial_distance:.4f}")
    
    # Display volumes
    display_cmyk_volumes(sample_CMYK_norm[0], max_well_volume, "Initial Sample Volumes")
    
    # Compare with target
    compare_colors(sample_CMYK_norm[0], target_cmyk, ["Initial Sample", "Target"])
    
    # Initialize BO variables exactly as in the original code
    X = sample_CMYK_norm  # Initial mixtures (inputs)
    y = -np.linalg.norm(X - target_cmyk, axis=1)  # Objective: minimize distance to target color
    
    # Gaussian Process Regression model - exactly as in original code
    kernel = C(constant_value=1, constant_value_bounds=(1e-3, 1e3)) * RBF(length_scale=1, length_scale_bounds=(1e-3, 1e3))
    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, alpha=1e-4, normalize_y=False)
    
    # Expected Improvement function - exactly as in original code
    def expected_improvement(X, X_sample, y_sample, model):
        """Return the expected improvement."""
        mu, sigma = model.predict(X, return_std=True)
        
        optiminal_sample = np.min(y_sample)
        
        with np.errstate(divide="ignore"):
            imp = mu - optiminal_sample - hyperparameter
            Z = imp / sigma
            ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
            ei[sigma == 0.0] = 0.0
        return ei.flatten()
    
    # Propose location function - exactly as in original code
    def propose_location(acquisition, X_sample, y_sample, bounds, n_restarts=25):
        dim = X_sample.shape[1]

        def min_obj(x):
            return -acquisition(x.reshape(-1, dim), X_sample, y_sample, gp)

        # Starting points for optimization
        x0_list = np.random.uniform(0, 1, size=(n_restarts, dim))
        best_x, best_val = None, float("inf")
        for x0 in x0_list:
            res = minimize(min_obj, x0, bounds=bounds, method="L-BFGS-B")
            if res.fun < best_val:
                best_val = res.fun
                best_x = res.x
        return best_x
    
    # Interactive simulation
    distances = [-y[0]]  # Track distances to target
    X_samples = [X[0]]  # Track all CMYK samples
    
    # Main interactive loop
    for i in range(max_iterations):
        # Fit GP model - exactly as in original code
        gp.fit(X, y)
        
        # Propose next sample - exactly as in original code
        X_next = propose_location(expected_improvement, X, y, bounds)
        
        # Calculate objective for new sample - exactly as in original code
        y_next = -np.linalg.norm(X_next - target_cmyk)
        
        # Display information for this iteration
        print(f"\n=== Iteration {i+1} ===")
        print(f"Suggested CMYK: [{', '.join([f'{v:.3f}' for v in X_next])}]")
        
        # Display volumes
        display_cmyk_volumes(X_next, max_well_volume, f"Iteration {i+1} Volumes")
        
        # Compare with target
        compare_colors(X_next, target_cmyk, [f"Iteration {i+1}", "Target"])
        
        print(f"Distance to target: {-y_next:.4f}")
        
        # Track the distance
        distances.append(-y_next)
        X_samples.append(X_next)
        
        # Check if we've reached the threshold
        if -y_next < threshold:
            print(f"\nReached threshold! Distance {-y_next:.4f} < {threshold}")
            break
        
        # Ask if user wants to continue
        user_input = input("\nContinue to next iteration? (y/n): ").strip().lower()
        if not user_input.startswith('y'):
            break
        
        # Append the new sample to dataset - exactly as in original code
        X = np.vstack((X, X_next))
        y = np.append(y, y_next)
    
    # Display final results
    print("\n=== Final Results ===")
    optimal_idx = np.argmin(distances)
    best_cmyk = X_samples[optimal_idx]
    
    print(f"Total iterations: {len(distances) - 1}")  # Subtract 1 for initial sample
    print(f"Best distance to target: {distances[optimal_idx]:.4f} at iteration {optimal_idx if optimal_idx > 0 else 'initial'}")
    print(f"Optimal CMYK mixture: [{', '.join([f'{v:.3f}' for v in best_cmyk])}]")
    
    # Display best volumes
    display_cmyk_volumes(best_cmyk, max_well_volume, "Optimal Sample Volumes")
    
    # Compare best with target
    compare_colors(best_cmyk, target_cmyk, ["Best Result", "Target"])
    
    # Plot convergence
    plt.figure(figsize=(10, 6))
    plt.plot(range(len(distances)), distances, marker='o', linestyle='-', color='b')
    plt.title('Convergence of Bayesian Optimization')
    plt.xlabel('Iteration')
    plt.ylabel('Distance to Target Color')
    plt.grid(True)
    plt.show()
    
    # Visualize all colors
    print("\nVisualizing all color samples...")
    visualize_color_selection(X_samples, target_cmyk)
    
    return best_cmyk, target_cmyk, distances

if __name__ == "__main__":
    print("=== Interactive Bayesian Optimization Color Matcher ===")
    print("This script uses the EXACT same BO implementation as your original code")
    print("It simulates the optimization process step by step, showing CMYK values at each iteration")
    
    # Set random seed for reproducibility
    np.random.seed(41)  # Same as in your original code
    
    run_interactive_bo_simulation()